Import the Data

In [15]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [16]:
from gensim.models import KeyedVectors
from pathlib import Path
import pandas as pd
import numpy as np
import os
import textwrap
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.base import BaseEstimator, TransformerMixin


In [17]:
model_path = "word2vec/GoogleNews-vectors-negative300.bin"
wv = KeyedVectors.load_word2vec_format(model_path, binary=True)
print("Loaded successfully")
print("Vector dimension:", wv.vector_size)
print("Example vector (king):", wv["king"][:10])

Loaded successfully
Vector dimension: 300
Example vector (king): [ 0.12597656  0.02978516  0.00860596  0.13964844 -0.02563477 -0.03613281
  0.11181641 -0.19824219  0.05126953  0.36328125]


In [18]:
data_dir = 'data_readinglevel'
x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))

x_test_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))

N, n_cols = x_train_df.shape
print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
print("Shape of y_train_df: %s" % str(y_train_df.shape))

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)


In [19]:
# Create the binary classification column y
# 2-3 is labeled as 0
# 4-5 is labeled as 1

y_train_df["true_labels"] = np.where(y_train_df["Coarse Label"].str.contains("Key Stage 2-3"), 0, 1)
y = y_train_df.true_labels.to_numpy() 

Drop Columns 

In [20]:
cols_to_remove = ['author', 'title', 'passage_id']
x_train_new = x_train_df.drop(columns = cols_to_remove)
x_test_new = x_test_df.drop(columns = cols_to_remove)

Embedding Function

In [21]:
class Word2VecEmbedder(BaseEstimator, TransformerMixin):
    def __init__(self, wv):
        self.wv = wv
        self.dim = wv.vector_size
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        vectors = []
        
        for text in X:
            words = text.split()
            word_vecs = [
                self.wv[w] for w in words
                if w in self.wv
            ]
            
            if len(word_vecs) == 0:
                vectors.append(np.zeros(self.dim))
            else:
                vectors.append(np.mean(word_vecs, axis=0))
                
        return np.array(vectors)

In [23]:
feature_columns = [
    "avg_word_length",
    "avg_sentence_length",
    "info_type_token_ratio",
    "readability_FleschReadingEase",
    "readability_DaleChallIndex", "punctuation_frequency"
]

text_columns = None

preprocessor = make_preprocessor(text_columns, feature_columns)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])